# Add DART-derived atmospheric perturbations to reference states

This notebook constructs 20-member atmospheric ensembles for configured reference cases and date windows. EN01 is an exact unperturbed control from the reference case. A selected 20-member DART subset defines the perturbation mean, and the first 19 rows of that subset provide perturbations for EN02-EN20:

```text
2011-12-16-00000, 2011-12-21-00000, 2011-12-26-00000, 2011-12-31-00000
2012-05-16-00000, 2012-05-21-00000, 2012-05-26-00000, 2012-05-31-00000
```

For selected DART subset \(S\) and member \(s_i\in S\),

\[
\delta A_{s_i} = A_{\mathrm{DART},s_i} - \overline{A}_{\mathrm{DART},S}
\]

and the reference atmosphere is perturbed as

\[
A_{\mathrm{output},i+1} = A_{\mathrm{reference}} + \alpha \delta A_{s_i},
\]

The selected 20-member subset is loaded from:

```text
/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection/2012-05-16-00000/selected_20_members.csv
```

The HYB perturbable atmospheric targets come from `DART_INIT/ATM`. The FREE_CLIM reference state is read from `F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM/archive/rest/<date>/*.eam.i.<date>.nc` and written as `eam.i` initial-condition files. Matching non-atmosphere component restart files are not modified and are linked into each date's initial-condition library.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from netCDF4 import Dataset

try:
    display
except NameError:
    def display(obj):
        print(obj)

# -------------------------------------------------------------------
# User configuration: multi-reference perturbation libraries
# -------------------------------------------------------------------

ATM_VARS_TO_PERTURB = ["T", "U", "V", "Q", "PS"]
SELECTED_DART_MEMBER_CSV = Path(
    "/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/"
    "dart_subset_selection/2012-05-16-00000/selected_20_members.csv"
)
CONTROL_OUTPUT_MEMBER = 1
N_SELECTED_DART_MEMBERS = 20
N_PERTURBED_OUTPUTS = 19

ALPHA = 1.0
CLIP_Q = True
Q_MIN = 0.0

# Date-specific HYB eam.i targets compatible with the DART atmospheric anomalies.
HYB_REF_ATM_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM")
# Non-atmosphere component restart sets used by the HYB initial conditions.
HYB_REF_REST_ROOT = Path(
    "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
    "20241109.v3-LR.DATM.ne30pg2_r05_IcoswISC30E3r5.pm-cpu/archive/rest"
)

FREE_CLIM_CASE_PREFIX = "F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM"
FREE_CLIM_REF_REST_ROOT = Path(
    "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
    f"{FREE_CLIM_CASE_PREFIX}/archive/rest"
)

DATE_SETS = [
    {
        "set_name": "dec2011",
        "dates": ["2011-12-16-00000", "2011-12-21-00000", "2011-12-26-00000", "2011-12-31-00000"],
        "dart_case_prefix": "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "dart_rest_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/rest"
        ),
        "dart_diag_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40"
        ),
    },
    {
        "set_name": "may2012",
        "dates": ["2012-05-16-00000", "2012-05-21-00000", "2012-05-26-00000", "2012-05-31-00000"],
        "dart_case_prefix": "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
        "dart_rest_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/rest"
        ),
        "dart_diag_root": Path(
            "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/"
            "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40"
        ),
    },
]

REFERENCE_CASES = [
    {
        "ref_name": "HYB",
        "ref_atm_source": "hyb_eam_i",
        "ref_atm_dir": HYB_REF_ATM_DIR,
        "ref_rest_root": HYB_REF_REST_ROOT,
        "output_root": Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_dartpert20_atm_multidate"),
        "out_case_prefix": "HYB_DARTPERT20",
        "out_atm_stream": "eam.i",
    },
    {
        "ref_name": "FREE_CLIM",
        "ref_atm_source": "free_clim_eam_i",
        "ref_case_prefix": FREE_CLIM_CASE_PREFIX,
        "ref_rest_root": FREE_CLIM_REF_REST_ROOT,
        "output_root": Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate"),
        "out_case_prefix": "FREECLIM_DARTPERT20",
        "out_atm_stream": "eam.i",
    },
]

# Optional run selectors. Use one month or reference at a time for separate runs.
# Lists preserve order, e.g. RUN_REFERENCE_NAMES = ["FREE_CLIM", "HYB"].
# Sets/strings select names but keep the configured order.
RUN_DATE_SET_NAMES = 'may2012'  # None
RUN_REFERENCE_NAMES = 'HYB' #None

WRITE_OUTPUT_FILES = True
# Set True when rerunning after changing perturbation settings such as ALPHA or Q_MIN.
OVERWRITE_OUTPUT_FILES = True
RESUME_EXISTING_OUTPUTS = True
CREATE_COMPONENT_LINKS = True
COMPONENT_LINK_MODE = "symlink"  # "symlink" or "copy"

ENSEMBLE_MEAN_TOLERANCE = {
    "T": 1.0e-5,
    "U": 1.0e-5,
    "V": 1.0e-5,
    "Q": 1.0e-10,
    "PS": 1.0e-2,
}
PHYSICAL_LIMITS = {
    "T": (150.0, 400.0),
    "U": (-300.0, 300.0),
    "V": (-300.0, 300.0),
    "Q": (0.0, 0.1),
    "PS": (4.0e4, 1.2e5),
}
WARN_Q_CLIP_FRACTION = 1.0e-2
ENFORCE_MAX_Q_CLIP_FRACTION = False
MAX_Q_CLIP_FRACTION = 2.0e-2
DIMENSION_ALIASES = {"ncol_d": "ncol"}
GRID_COORDINATE_ATOL_DEGREES = 2.0e-5
FLOAT32_COORDINATE_RTOL = 1.0e-6
FLOAT32_COORDINATE_ATOL = 1.0e-7
COMPONENT_PATTERNS = [
    "*.elm.r.*.nc",
    "*.mosart.r.*.nc",
    "*.mpaso.rst.*.nc",
    "*.mpassi.rst.*.nc",
    "*.cpl.r.*.nc",
    "rpointer.drv",
    "rpointer.ice",
    "rpointer.lnd",
    "rpointer.ocn",
    "rpointer.rof",
]

def selected_names_or_all(requested, available, label, preserve_requested_order=False):
    available = list(available)
    available_set = set(available)
    if requested is None:
        return available
    if isinstance(requested, str):
        requested = [requested]
    requested = list(requested)
    unknown = set(requested) - available_set
    if unknown:
        raise ValueError(
            f"Unknown {label}: {sorted(unknown)}. Available: {sorted(available_set)}"
        )
    if preserve_requested_order:
        return requested
    return [name for name in available if name in set(requested)]

active_date_set_names = selected_names_or_all(
    RUN_DATE_SET_NAMES,
    [date_set["set_name"] for date_set in DATE_SETS],
    "date set names",
    preserve_requested_order=isinstance(RUN_DATE_SET_NAMES, (list, tuple)),
)
active_reference_names = selected_names_or_all(
    RUN_REFERENCE_NAMES,
    [ref_case["ref_name"] for ref_case in REFERENCE_CASES],
    "reference names",
    preserve_requested_order=isinstance(RUN_REFERENCE_NAMES, (list, tuple)),
)
DATE_SETS_BY_NAME = {date_set["set_name"]: date_set for date_set in DATE_SETS}
REFERENCE_CASES_BY_NAME = {ref_case["ref_name"]: ref_case for ref_case in REFERENCE_CASES}
DATE_SETS = [DATE_SETS_BY_NAME[name] for name in active_date_set_names]
REFERENCE_CASES = [REFERENCE_CASES_BY_NAME[name] for name in active_reference_names]
if not DATE_SETS:
    raise ValueError("No DATE_SETS selected.")
if not REFERENCE_CASES:
    raise ValueError("No REFERENCE_CASES selected.")

for ref_case in REFERENCE_CASES:
    ref_case["output_root"].mkdir(parents=True, exist_ok=True)

print("Reference cases:")
for ref_case in REFERENCE_CASES:
    print(
        ref_case["ref_name"],
        "source=", ref_case["ref_atm_source"],
        "ref_rest_root=", ref_case["ref_rest_root"],
        "output_root=", ref_case["output_root"].resolve(),
    )
for date_set in DATE_SETS:
    print(date_set["set_name"], date_set["dates"])


Reference cases:
HYB source= hyb_eam_i ref_rest_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/20241109.v3-LR.DATM.ne30pg2_r05_IcoswISC30E3r5.pm-cpu/archive/rest output_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_dartpert20_atm_multidate
FREE_CLIM source= free_clim_eam_i ref_rest_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM/archive/rest output_root= /global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate
dec2011 ['2011-12-16-00000', '2011-12-21-00000', '2011-12-26-00000', '2011-12-31-00000']
may2012 ['2012-05-16-00000', '2012-05-21-00000', '2012-05-26-00000', '2012-05-31-00000']


## 1. Load the selected DART members and define output mapping


In [2]:
def load_selected_dart_members(csv_path: Path, expected_count: int) -> list[int]:
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Selected-member CSV not found: {csv_path}. "
            "Run the 20-member DART subset-selection notebook first."
        )
    table = pd.read_csv(csv_path)
    if "member" in table.columns:
        members = table["member"].astype(int).tolist()
    elif "member_label" in table.columns:
        members = [int(str(value).upper().removeprefix("EN")) for value in table["member_label"]]
    else:
        raise ValueError("Selected-member CSV requires 'member' or 'member_label'.")
    if len(members) != len(set(members)):
        raise ValueError("Selected-member CSV contains duplicate members.")
    if len(members) != expected_count:
        raise ValueError(f"Expected {expected_count} selected members, found {len(members)}.")
    invalid = [member for member in members if member < 1 or member > 40]
    if invalid:
        raise ValueError(f"Selected DART members outside EN01-EN40: {invalid}")
    return members

selected_dart_members = load_selected_dart_members(
    SELECTED_DART_MEMBER_CSV, N_SELECTED_DART_MEMBERS
)
centering_dart_members = selected_dart_members
perturbation_dart_members = selected_dart_members[:N_PERTURBED_OUTPUTS]
omitted_dart_members = selected_dart_members[N_PERTURBED_OUTPUTS:]
if len(omitted_dart_members) != 1:
    raise ValueError("Exactly one selected DART source member must be omitted.")
perturbed_output_members = list(range(2, 2 + N_PERTURBED_OUTPUTS))
output_to_dart_member = dict(zip(perturbed_output_members, perturbation_dart_members))
all_output_members = [CONTROL_OUTPUT_MEMBER] + perturbed_output_members
if all_output_members != list(range(1, 21)):
    raise ValueError("Final output ensemble must be EN01-EN20.")

member_mapping = pd.DataFrame({
    "output_member": perturbed_output_members,
    "output_member_label": [f"EN{m:02d}" for m in perturbed_output_members],
    "dart_member": perturbation_dart_members,
    "dart_member_label": [f"EN{m:02d}" for m in perturbation_dart_members],
})
complete_member_mapping = pd.concat([
    pd.DataFrame([{
        "output_member": CONTROL_OUTPUT_MEMBER,
        "output_member_label": "EN01",
        "dart_member": pd.NA,
        "dart_member_label": "CONTROL",
    }]),
    member_mapping,
], ignore_index=True)
print("Control output member: EN01")
print("DART centering members:", centering_dart_members)
print("Omitted DART source member:", omitted_dart_members[0])
display(complete_member_mapping)


Control output member: EN01
DART centering members: [2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 24, 26, 27, 28, 31, 32, 34, 38]
Omitted DART source member: 38


,output_member,output_member_label,dart_member,dart_member_label
0,1,EN01,<NA>,CONTROL
1,2,EN02,2,EN02
2,3,EN03,3,EN03
3,4,EN04,5,EN05
4,5,EN05,7,EN07
5,6,EN06,11,EN11
6,7,EN07,12,EN12
7,8,EN08,13,EN13
8,9,EN09,14,EN14
9,10,EN10,15,EN15


## 2. Build per-date configuration table


In [3]:
def ref_atm_file(ref_case: dict, date_tag: str) -> Path:
    if ref_case["ref_atm_source"] == "hyb_eam_i":
        return ref_case["ref_atm_dir"] / f"v3.LR.ne30pg2.ERA5.eam.i.{date_tag}.nc"
    if ref_case["ref_atm_source"] == "free_clim_eam_i":
        return (
            ref_case["ref_rest_root"]
            / date_tag
            / f"{ref_case['ref_case_prefix']}.eam.i.{date_tag}.nc"
        )
    raise ValueError(f"Unknown reference atmosphere source: {ref_case['ref_atm_source']}")

def ref_rest_dir(ref_case: dict, date_tag: str) -> Path:
    return ref_case["ref_rest_root"] / date_tag

def output_dir_for(ref_case: dict, set_name: str, date_tag: str) -> Path:
    return ref_case["output_root"] / date_tag

def dart_rest_dir(date_set: dict, date_tag: str) -> Path:
    return date_set["dart_rest_root"] / date_tag

def dart_diag_dir(date_set: dict, date_tag: str) -> Path:
    return date_set["dart_diag_root"] / date_tag

def dart_rest_member_file(date_set: dict, date_tag: str, member: int) -> Path:
    prefix = date_set["dart_case_prefix"]
    return dart_rest_dir(date_set, date_tag) / f"{prefix}.EN{member:02d}.eam.i.{date_tag}.nc"

def dart_forecast_member_file(date_set: dict, date_tag: str, member: int) -> Path:
    prefix = date_set["dart_case_prefix"]
    return dart_diag_dir(date_set, date_tag) / f"{prefix}.eam_{member:04d}.e.forecast.{date_tag}"

def dart_forecast_mean_file(date_set: dict, date_tag: str) -> Path:
    prefix = date_set["dart_case_prefix"]
    return dart_diag_dir(date_set, date_tag) / f"{prefix}.dart.e.eam_forecast_mean.{date_tag}.nc"

def has_usable_vars(path: Path, variables) -> bool:
    if not path.is_file():
        return False
    try:
        with xr.open_dataset(path, decode_times=False) as ds:
            for var in variables:
                if var not in ds:
                    return False
                if any(size == 0 for size in ds[var].shape):
                    return False
    except Exception:
        return False
    return True

def choose_dart_source(date_set: dict, date_tag: str) -> dict:
    rest_files = {m: dart_rest_member_file(date_set, date_tag, m) for m in selected_dart_members}
    rest_ok = all(has_usable_vars(path, ATM_VARS_TO_PERTURB) for path in rest_files.values())
    if rest_ok:
        return {
            "source_type": "rest_eam_i",
            "member_files": rest_files,
            "mean_file": None,
            "mean_mode": "compute_from_members",
        }

    forecast_files = {m: dart_forecast_member_file(date_set, date_tag, m) for m in selected_dart_members}
    forecast_ok = all(has_usable_vars(path, ATM_VARS_TO_PERTURB) for path in forecast_files.values())
    if forecast_ok:
        return {
            "source_type": "dart_en40_forecast",
            "member_files": forecast_files,
            "mean_file": None,
            "mean_mode": "compute_selected_subset_mean",
        }

    raise FileNotFoundError(f"No usable DART atmospheric source found for {date_tag}")

run_configs = []
for ref_case in REFERENCE_CASES:
    for date_set in DATE_SETS:
        for date_tag in date_set["dates"]:
            source = choose_dart_source(date_set, date_tag)
            cfg = {
                "ref_name": ref_case["ref_name"],
                "ref_atm_source": ref_case["ref_atm_source"],
                "set_name": date_set["set_name"],
                "date_tag": date_tag,
                "dart_case_prefix": date_set["dart_case_prefix"],
                "dart_rest_dir": dart_rest_dir(date_set, date_tag),
                "dart_diag_dir": dart_diag_dir(date_set, date_tag),
                "dart_source_type": source["source_type"],
                "dart_member_files": source["member_files"],
                "dart_mean_file": source["mean_file"],
                "dart_mean_mode": source["mean_mode"],
                "ref_atm_file": ref_atm_file(ref_case, date_tag),
                "ref_rest_dir": ref_rest_dir(ref_case, date_tag),
                "output_root": ref_case["output_root"],
                "output_dir": output_dir_for(ref_case, date_set["set_name"], date_tag),
                "out_case_prefix": ref_case["out_case_prefix"],
                "out_atm_stream": ref_case["out_atm_stream"],
            }
            run_configs.append(cfg)

config_table = pd.DataFrame([
    {
        "ref_name": cfg["ref_name"],
        "set": cfg["set_name"],
        "date": cfg["date_tag"],
        "dart_source_type": cfg["dart_source_type"],
        "dart_mean_mode": cfg["dart_mean_mode"],
        "ref_atm_exists": cfg["ref_atm_file"].is_file(),
        "ref_rest_exists": cfg["ref_rest_dir"].is_dir(),
        "output_dir": str(cfg["output_dir"]),
    }
    for cfg in run_configs
])
display(config_table)

missing_ref = [
    f"{cfg['ref_name']} {cfg['date_tag']}"
    for cfg in run_configs
    if not cfg["ref_atm_file"].is_file() or not cfg["ref_rest_dir"].is_dir()
]
if missing_ref:
    raise FileNotFoundError("Missing reference files/directories for " + ", ".join(missing_ref))


,ref_name,set,date,dart_source_type,dart_mean_mode,ref_atm_exists,ref_rest_exists,output_dir
0,HYB,dec2011,2011-12-16-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
1,HYB,dec2011,2011-12-21-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
2,HYB,dec2011,2011-12-26-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
3,HYB,dec2011,2011-12-31-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
4,HYB,may2012,2012-05-16-00000,rest_eam_i,compute_from_members,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
5,HYB,may2012,2012-05-21-00000,rest_eam_i,compute_from_members,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
6,HYB,may2012,2012-05-26-00000,rest_eam_i,compute_from_members,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
7,HYB,may2012,2012-05-31-00000,rest_eam_i,compute_from_members,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
8,FREE_CLIM,dec2011,2011-12-16-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...
9,FREE_CLIM,dec2011,2011-12-21-00000,dart_en40_forecast,compute_selected_subset_mean,True,True,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free...


## 3. Compatibility and alignment helpers


In [4]:
def inventory(path: Path, variables):
    rows = []
    with xr.open_dataset(path, decode_times=False) as ds:
        for var in variables:
            if var not in ds:
                rows.append({"variable": var, "status": "missing", "dims": "", "shape": "", "dtype": "", "units": "", "has_empty_dim": True})
            else:
                da = ds[var]
                rows.append({
                    "variable": var,
                    "status": "found",
                    "dims": str(da.dims),
                    "shape": str(da.shape),
                    "dtype": str(da.dtype),
                    "units": da.attrs.get("units", ""),
                    "has_empty_dim": any(s == 0 for s in da.shape),
                })
    return pd.DataFrame(rows)

def ref_coords_for(ref_da):
    return {dim: ref_da.coords[dim] for dim in ref_da.dims if dim in ref_da.coords}

def validate_alignment(da, ref_da, label):
    rename_dims = {
        dim: DIMENSION_ALIASES[dim]
        for dim in da.dims
        if dim in DIMENSION_ALIASES
        and DIMENSION_ALIASES[dim] in ref_da.dims
        and DIMENSION_ALIASES[dim] not in da.dims
    }
    normalized = da.rename(rename_dims)
    if set(normalized.dims) != set(ref_da.dims):
        raise ValueError(
            f"{label}: DART dims={da.dims}, REF dims={ref_da.dims}"
        )

    aligned = normalized.transpose(*ref_da.dims)
    if aligned.shape != ref_da.shape:
        raise ValueError(
            f"{label}: shape after transpose={aligned.shape}, REF={ref_da.shape}"
        )

    for dim in ref_da.dims:
        if dim not in aligned.coords or dim not in ref_da.coords:
            continue
        if aligned.sizes[dim] == 1 and ref_da.sizes[dim] == 1:
            continue
        dart_coord = np.asarray(aligned.coords[dim].values)
        ref_coord = np.asarray(ref_da.coords[dim].values)
        if dart_coord.shape != ref_coord.shape:
            raise ValueError(f"{label}: coordinate size mismatch for {dim}")
        if np.issubdtype(dart_coord.dtype, np.number) and np.issubdtype(ref_coord.dtype, np.number):
            uses_float32 = (
                np.issubdtype(dart_coord.dtype, np.floating) and dart_coord.dtype.itemsize <= 4
            ) or (
                np.issubdtype(ref_coord.dtype, np.floating) and ref_coord.dtype.itemsize <= 4
            )
            rtol = FLOAT32_COORDINATE_RTOL if uses_float32 else 1.0e-10
            atol = FLOAT32_COORDINATE_ATOL if uses_float32 else 1.0e-12
            matches = np.allclose(
                dart_coord, ref_coord, rtol=rtol, atol=atol, equal_nan=True
            )
        else:
            matches = np.array_equal(dart_coord, ref_coord)
        if not matches:
            raise ValueError(f"{label}: coordinate mismatch for {dim}")
    return aligned

def validate_horizontal_grid(dart_ds, ref_ds, label):
    if "ncol_d" not in dart_ds.sizes or "ncol" not in ref_ds.sizes:
        return
    if dart_ds.sizes["ncol_d"] != ref_ds.sizes["ncol"]:
        raise ValueError(
            f"{label}: ncol_d={dart_ds.sizes['ncol_d']} != "
            f"reference ncol={ref_ds.sizes['ncol']}"
        )
    required = [(dart_ds, "lat_d"), (dart_ds, "lon_d"), (ref_ds, "lat"), (ref_ds, "lon")]
    missing = [name for ds, name in required if name not in ds]
    if missing:
        raise ValueError(f"{label}: missing grid coordinates {missing}")
    dart_lat = np.asarray(dart_ds["lat_d"].values)
    dart_lon = np.asarray(dart_ds["lon_d"].values)
    ref_lat = np.asarray(ref_ds["lat"].values)
    ref_lon = np.asarray(ref_ds["lon"].values)
    if not np.allclose(
        dart_lat, ref_lat, rtol=0.0, atol=GRID_COORDINATE_ATOL_DEGREES
    ):
        raise ValueError(f"{label}: latitude grid mismatch")
    wrapped_lon_difference = (dart_lon - ref_lon + 180.0) % 360.0 - 180.0
    if not np.allclose(
        wrapped_lon_difference, 0.0, rtol=0.0, atol=GRID_COORDINATE_ATOL_DEGREES
    ):
        raise ValueError(f"{label}: longitude grid mismatch")

def align_to_ref_dims(da, ref_da):
    aligned = validate_alignment(da, ref_da, da.name or "unnamed variable")
    return aligned.assign_coords(ref_coords_for(ref_da))

def load_aligned_var(path: Path, var: str, ref_ds):
    with xr.open_dataset(path, decode_times=False, mask_and_scale=True, cache=False) as ds:
        da = ds[var].load().astype("float64")
    return align_to_ref_dims(da, ref_ds[var])

def validate_all_member_structures(cfg):
    errors = []
    with xr.open_dataset(
        cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False
    ) as ref_ds:
        missing_ref = [var for var in ATM_VARS_TO_PERTURB if var not in ref_ds]
        if missing_ref:
            errors.append(f"Reference missing variables: {missing_ref}")

        for member in selected_dart_members:
            path = cfg["dart_member_files"][member]
            try:
                with xr.open_dataset(
                    path, decode_times=False, mask_and_scale=True, cache=False
                ) as dart_ds:
                    validate_horizontal_grid(dart_ds, ref_ds, f"EN{member:02d}")
                    for var in ATM_VARS_TO_PERTURB:
                        if var not in ref_ds or var not in dart_ds:
                            if var not in dart_ds:
                                errors.append(f"EN{member:02d} missing {var}: {path}")
                            continue
                        if not np.issubdtype(dart_ds[var].dtype, np.number):
                            errors.append(f"EN{member:02d} {var} is not numeric: {path}")
                            continue
                        try:
                            validate_alignment(
                                dart_ds[var], ref_ds[var], f"EN{member:02d} {var}"
                            )
                        except ValueError as exc:
                            errors.append(str(exc))
            except Exception as exc:
                errors.append(f"Could not inspect EN{member:02d} {path}: {exc}")

    if errors:
        raise ValueError(
            "DART member compatibility check failed:\n" + "\n".join(errors)
        )
    return list(ATM_VARS_TO_PERTURB)

def assert_finite(da, label):
    values = np.asarray(da.values)
    n_nan = int(np.isnan(values).sum())
    n_inf = int(np.isinf(values).sum())
    if n_nan or n_inf:
        raise ValueError(f"{label} contains {n_nan} NaNs and {n_inf} infinities")

def compute_member_mean(member_files, variables, ref_file):
    mean_vars = {}
    with xr.open_dataset(ref_file, decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in variables:
            print(f"Computing selected 20-member DART mean for {var}")
            running = np.zeros(ref_ds[var].shape, dtype=np.float64)
            for m in selected_dart_members:
                da = load_aligned_var(member_files[m], var, ref_ds)
                assert_finite(da, f"EN{m:02d} {var}")
                running += da.values
            mean_vars[var] = xr.DataArray(
                running / len(selected_dart_members),
                dims=ref_ds[var].dims,
                coords=ref_coords_for(ref_ds[var]),
                attrs=dict(ref_ds[var].attrs),
                name=var,
            )
            assert_finite(mean_vars[var], f"DART ensemble mean {var}")
    return xr.Dataset(mean_vars)

def load_dart_mean(cfg, variables):
    if cfg["dart_mean_mode"] in {
        "compute_from_members", "compute_selected_subset_mean"
    }:
        return compute_member_mean(cfg["dart_member_files"], variables, cfg["ref_atm_file"])
    raise ValueError(f"Unknown mean mode: {cfg['dart_mean_mode']}")

compat_rows = []
for cfg in run_configs:
    print(f"Preflighting all 20 selected members for {cfg['date_tag']}")
    cfg["compatible_vars"] = validate_all_member_structures(cfg)
    compat_rows.append({
        "set": cfg["set_name"],
        "date": cfg["date_tag"],
        "dart_source_type": cfg["dart_source_type"],
        "compatible_vars": ",".join(cfg["compatible_vars"]),
    })
display(pd.DataFrame(compat_rows))


Preflighting all 20 selected members for 2011-12-16-00000
Preflighting all 20 selected members for 2011-12-21-00000
Preflighting all 20 selected members for 2011-12-26-00000
Preflighting all 20 selected members for 2011-12-31-00000
Preflighting all 20 selected members for 2012-05-16-00000
Preflighting all 20 selected members for 2012-05-21-00000
Preflighting all 20 selected members for 2012-05-26-00000
Preflighting all 20 selected members for 2012-05-31-00000
Preflighting all 20 selected members for 2011-12-16-00000


ValueError: DART member compatibility check failed:
Could not inspect EN02 /global/cfs/cdirs/m4849/zhan391/e3sm_dart/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40/2011-12-16-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.eam_0002.e.forecast.2011-12-16-00000: EN02: ncol_d=48602 != reference ncol=21600
Could not inspect EN03 /global/cfs/cdirs/m4849/zhan391/e3sm_dart/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40/2011-12-16-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.eam_0003.e.forecast.2011-12-16-00000: EN03: ncol_d=48602 != reference ncol=21600
Could not inspect EN05 /global/cfs/cdirs/m4849/zhan391/e3sm_dart/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en40/2011-12-16-00000/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.eam_0005.e.forecast.2011-12-16-00000: EN05: ncol_d=48602 != reference ncol=21600
Could not inspect EN07 /global/cfs/cdirs/m4849/zhan391/e3sm_dart/DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/arch

## 4. Diagnostics, writing, and validation functions


In [ ]:
def output_atm_file(cfg, member: int) -> Path:
    return cfg["output_dir"] / f"{cfg['out_case_prefix']}.EN{member:02d}.{cfg['out_atm_stream']}.{cfg['date_tag']}.nc"

def assert_no_new_invalid(reference, candidate, label):
    ref_valid = np.isfinite(np.asarray(reference.values))
    candidate_valid = np.isfinite(np.asarray(candidate.values))
    newly_invalid = ref_valid & ~candidate_valid
    if np.any(newly_invalid):
        raise ValueError(
            f"{label} introduced {int(newly_invalid.sum())} new invalid values"
        )

def enforce_physical_limits(var, da, label):
    lower, upper = PHYSICAL_LIMITS[var]
    actual_min = float(da.min(skipna=True).item())
    actual_max = float(da.max(skipna=True).item())
    if (
        not np.isfinite(actual_min)
        or not np.isfinite(actual_max)
        or actual_min < lower
        or actual_max > upper
    ):
        raise ValueError(
            f"{label} outside broad physical limits: min={actual_min}, "
            f"max={actual_max}, allowed=({lower}, {upper})"
        )

def perturbation_diagnostics(cfg, dart_mean):
    rows = []
    with xr.open_dataset(cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in cfg["compatible_vars"]:
            member_mean_delta = []
            member_rms_delta = []
            assert_finite(dart_mean[var], f"DART ensemble mean {var}")
            for m in selected_dart_members:
                dart_var = load_aligned_var(cfg["dart_member_files"][m], var, ref_ds)
                assert_finite(dart_var, f"EN{m:02d} {var}")
                perturb_da = (dart_var - dart_mean[var]).load()
                assert_finite(perturb_da, f"EN{m:02d} {var} perturbation")
                perturb = perturb_da.values
                member_mean_delta.append(float(np.mean(perturb)))
                member_rms_delta.append(float(np.sqrt(np.mean(perturb ** 2))))
            rows.append({
                "ref_name": cfg["ref_name"],
                "date": cfg["date_tag"],
                "variable": var,
                "mean_of_member_mean_perturbations": float(np.mean(member_mean_delta)),
                "std_of_member_mean_perturbations": float(np.std(member_mean_delta, ddof=1)),
                "mean_member_rms_perturbation": float(np.mean(member_rms_delta)),
                "min_member_rms_perturbation": float(np.min(member_rms_delta)),
                "max_member_rms_perturbation": float(np.max(member_rms_delta)),
            })
    return pd.DataFrame(rows)

def compare_attribute_values(ref_value, out_value):
    try:
        return np.array_equal(
            np.asarray(ref_value), np.asarray(out_value), equal_nan=True
        )
    except TypeError:
        return np.array_equal(np.asarray(ref_value), np.asarray(out_value))

def validate_same_netcdf_structure(ref_file, out_file):
    errors = []
    with Dataset(ref_file, mode="r") as ref_nc, Dataset(out_file, mode="r") as out_nc:
        if ref_nc.data_model != out_nc.data_model:
            errors.append(f"Data model changed: {ref_nc.data_model} -> {out_nc.data_model}")

        ref_dims = list(ref_nc.dimensions)
        out_dims = list(out_nc.dimensions)
        if ref_dims != out_dims:
            errors.append(f"Dimension names/order changed: {ref_dims} -> {out_dims}")
        for dim_name in ref_dims:
            if dim_name not in out_nc.dimensions:
                continue
            ref_dim = ref_nc.dimensions[dim_name]
            out_dim = out_nc.dimensions[dim_name]
            if len(ref_dim) != len(out_dim):
                errors.append(f"Dimension {dim_name} size changed")
            if ref_dim.isunlimited() != out_dim.isunlimited():
                errors.append(f"Dimension {dim_name} unlimited status changed")

        ref_vars = list(ref_nc.variables)
        out_vars = list(out_nc.variables)
        if ref_vars != out_vars:
            errors.append("Variable names or ordering changed")

        ref_global_attrs = list(ref_nc.ncattrs())
        out_global_attrs = list(out_nc.ncattrs())
        if ref_global_attrs != out_global_attrs:
            errors.append("Global attribute names or ordering changed")
        for attr in ref_global_attrs:
            if attr in out_global_attrs and not compare_attribute_values(
                ref_nc.getncattr(attr), out_nc.getncattr(attr)
            ):
                errors.append(f"Global attribute {attr!r} changed")

        for var_name in ref_vars:
            if var_name not in out_nc.variables:
                continue
            ref_var = ref_nc.variables[var_name]
            out_var = out_nc.variables[var_name]
            structural_pairs = {
                "dtype": (str(ref_var.dtype), str(out_var.dtype)),
                "dimensions": (tuple(ref_var.dimensions), tuple(out_var.dimensions)),
                "shape": (tuple(ref_var.shape), tuple(out_var.shape)),
                "filters": (ref_var.filters(), out_var.filters()),
                "chunking": (ref_var.chunking(), out_var.chunking()),
                "endian": (ref_var.endian(), out_var.endian()),
                "attributes": (list(ref_var.ncattrs()), list(out_var.ncattrs())),
            }
            for property_name, (ref_value, out_value) in structural_pairs.items():
                if ref_value != out_value:
                    errors.append(f"{var_name} {property_name} changed")
            for attr in ref_var.ncattrs():
                if attr in out_var.ncattrs() and not compare_attribute_values(
                    ref_var.getncattr(attr), out_var.getncattr(attr)
                ):
                    errors.append(f"{var_name} attribute {attr!r} changed")

    if errors:
        raise RuntimeError(
            f"NetCDF structure validation failed for {out_file}:\n"
            + "\n".join(errors)
        )

def validate_unperturbed_variables_identical(ref_file, out_file, perturbed_variables):
    with Dataset(ref_file, mode="r") as ref_nc, Dataset(out_file, mode="r") as out_nc:
        for var_name in ref_nc.variables:
            if var_name in perturbed_variables:
                continue
            ref_var = ref_nc.variables[var_name]
            out_var = out_nc.variables[var_name]
            ref_var.set_auto_maskandscale(False)
            out_var.set_auto_maskandscale(False)
            if not np.array_equal(ref_var[:], out_var[:]):
                raise RuntimeError(f"Unperturbed variable {var_name} changed in {out_file}")

def report_variable_storage(ref_file, variables):
    with Dataset(ref_file, mode="r") as nc:
        for var_name in variables:
            nc_var = nc.variables[var_name]
            print(
                var_name,
                "dtype=", nc_var.dtype,
                "dimensions=", nc_var.dimensions,
                "chunking=", nc_var.chunking(),
                "filters=", nc_var.filters(),
                "scale_factor=", getattr(nc_var, "scale_factor", None),
                "add_offset=", getattr(nc_var, "add_offset", None),
                "_FillValue=", getattr(nc_var, "_FillValue", None),
            )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_control_file(cfg):
    ref_file = cfg["ref_atm_file"]
    out_file = output_atm_file(cfg, CONTROL_OUTPUT_MEMBER)
    if out_file.exists() and not OVERWRITE_OUTPUT_FILES:
        if not RESUME_EXISTING_OUTPUTS:
            raise FileExistsError(
                f"Output already exists: {out_file}. "
                "Set RESUME_EXISTING_OUTPUTS=True to validate and reuse it."
            )
        ref_checksum = sha256_file(ref_file)
        control_checksum = sha256_file(out_file)
        if control_checksum != ref_checksum:
            raise RuntimeError(
                f"Existing EN01 checksum does not match the reference: {out_file}"
            )
        print(f"Reusing checksum-verified control: {out_file}")
        return out_file, control_checksum
    tmp_file = out_file.with_suffix(out_file.suffix + ".tmp")
    if tmp_file.exists():
        tmp_file.unlink()
    try:
        shutil.copy2(ref_file, tmp_file)
        ref_checksum = sha256_file(ref_file)
        control_checksum = sha256_file(tmp_file)
        if control_checksum != ref_checksum:
            raise RuntimeError("EN01 control checksum does not match the reference.")
        tmp_file.replace(out_file)
    except Exception:
        if tmp_file.exists():
            tmp_file.unlink()
        raise
    return out_file, control_checksum

def write_member_file(cfg, output_member: int, dart_member: int, dart_mean):
    out_file = output_atm_file(cfg, output_member)
    ref_file = cfg["ref_atm_file"]
    if out_file.exists() and not OVERWRITE_OUTPUT_FILES:
        if not RESUME_EXISTING_OUTPUTS:
            raise FileExistsError(
                f"Output already exists: {out_file}. "
                "Set RESUME_EXISTING_OUTPUTS=True to validate and reuse it."
            )
        validate_same_netcdf_structure(ref_file, out_file)
        validate_unperturbed_variables_identical(
            ref_file, out_file, set(cfg["compatible_vars"])
        )
        prior_q_rows = []
        q_diag_file = cfg["output_dir"] / "q_clipping_diagnostics.csv"
        if q_diag_file.is_file() and q_diag_file.stat().st_size:
            prior_q = pd.read_csv(q_diag_file)
            if "output_member" in prior_q.columns:
                prior_q_rows = prior_q.loc[
                    prior_q["output_member"] == output_member
                ].to_dict(orient="records")
        print(f"Reusing structure-verified member: {out_file}")
        return out_file, prior_q_rows
    tmp_file = out_file.with_suffix(out_file.suffix + ".tmp")
    if tmp_file.exists():
        tmp_file.unlink()

    q_clip_rows = []
    updated_values = {}
    with xr.open_dataset(
        ref_file, decode_times=False, mask_and_scale=True, cache=False
    ) as ref_ds:
        for var in cfg["compatible_vars"]:
            ref_da = ref_ds[var].load().astype("float64")
            dart_var = load_aligned_var(cfg["dart_member_files"][dart_member], var, ref_ds)
            assert_finite(dart_var, f"DART EN{dart_member:02d} {var}")
            assert_finite(dart_mean[var], f"DART ensemble mean {var}")
            perturb = dart_var - dart_mean[var].astype("float64")
            assert_finite(perturb, f"DART EN{dart_member:02d} {var} perturbation")
            new_da = ref_da + ALPHA * perturb

            if var == "Q" and CLIP_Q:
                clip_mask = new_da < Q_MIN
                number_clipped = int(clip_mask.sum().item())
                fraction_clipped = float(clip_mask.mean().item())
                minimum_before_clip = float(new_da.min(skipna=True).item())
                maximum_q_correction = max(0.0, Q_MIN - minimum_before_clip)
                exceeds_q_clip_warning = fraction_clipped > WARN_Q_CLIP_FRACTION
                q_clip_rows.append({
                    "ref_name": cfg["ref_name"],
                    "date": cfg["date_tag"],
                    "output_member": output_member,
                    "output_member_label": f"EN{output_member:02d}",
                    "dart_member": dart_member,
                    "dart_member_label": f"EN{dart_member:02d}",
                    "number_q_clipped": number_clipped,
                    "fraction_q_clipped": fraction_clipped,
                    "minimum_q_before_clip": minimum_before_clip,
                    "maximum_q_correction": maximum_q_correction,
                    "exceeds_q_clip_warning": exceeds_q_clip_warning,
                })
                if exceeds_q_clip_warning:
                    print(
                        f"Warning: output EN{output_member:02d} from DART EN{dart_member:02d} "
                        f"Q clipping fraction {fraction_clipped:.6e} "
                        f"exceeds warning threshold {WARN_Q_CLIP_FRACTION:.6e}"
                    )
                if ENFORCE_MAX_Q_CLIP_FRACTION and fraction_clipped > MAX_Q_CLIP_FRACTION:
                    raise ValueError(
                        f"Output EN{output_member:02d} from DART EN{dart_member:02d} "
                        f"Q clipping fraction {fraction_clipped:.6e} "
                        f"exceeds hard limit {MAX_Q_CLIP_FRACTION:.6e}"
                    )
                new_da = new_da.clip(min=Q_MIN)

            if new_da.dims != ref_da.dims or new_da.shape != ref_da.shape:
                raise ValueError(
                    f"Output EN{output_member:02d} {var} changed structure: "
                    f"dims={new_da.dims}, shape={new_da.shape}"
                )
            assert_no_new_invalid(ref_da, new_da, f"output EN{output_member:02d} {var}")
            enforce_physical_limits(var, new_da, f"output EN{output_member:02d} {var}")
            updated_values[var] = np.asarray(new_da.values)

    try:
        shutil.copy2(ref_file, tmp_file)
        with Dataset(tmp_file, mode="r+") as nc:
            for var, values in updated_values.items():
                if var not in nc.variables:
                    raise KeyError(f"{var} missing from copied file {tmp_file}")
                nc_var = nc.variables[var]
                if nc_var.shape != values.shape:
                    raise ValueError(
                        f"{var}: NetCDF shape {nc_var.shape} != {values.shape}"
                    )
                nc_var[:] = values
            nc.sync()
        validate_same_netcdf_structure(ref_file, tmp_file)
        validate_unperturbed_variables_identical(
            ref_file, tmp_file, set(cfg["compatible_vars"])
        )
        tmp_file.replace(out_file)
    except Exception:
        if tmp_file.exists():
            tmp_file.unlink()
        raise
    return out_file, q_clip_rows

def validate_written_files(cfg, dart_mean):
    rows = []
    with xr.open_dataset(cfg["ref_atm_file"], decode_times=False, mask_and_scale=True, cache=False) as ref_ds:
        for var in cfg["compatible_vars"]:
            member_mean_delta = []
            member_rms_delta = []
            member_min_delta = []
            member_max_delta = []
            ens_sum = None
            for output_member in perturbed_output_members:
                with xr.open_dataset(output_atm_file(cfg, output_member), decode_times=False, mask_and_scale=True, cache=False) as out_ds:
                    delta = (out_ds[var] - ref_ds[var]).load()
                    vals = delta.values
                    assert_no_new_invalid(
                        ref_ds[var], out_ds[var],
                        f"written output EN{output_member:02d} {var}",
                    )
                    member_mean_delta.append(float(np.mean(vals)))
                    member_rms_delta.append(float(np.sqrt(np.mean(vals ** 2))))
                    member_min_delta.append(float(np.min(vals)))
                    member_max_delta.append(float(np.max(vals)))
                    ens_sum = delta.astype("float64") if ens_sum is None else ens_sum + delta.astype("float64")
            perturbed_mean_delta = ens_sum / len(perturbed_output_members)
            complete_mean_delta = ens_sum / len(all_output_members)
            omitted_member = omitted_dart_members[0]
            omitted_var = load_aligned_var(
                cfg["dart_member_files"][omitted_member], var, ref_ds
            )
            expected_complete_mean_delta = (
                -ALPHA * (omitted_var - dart_mean[var]) / len(all_output_members)
            )
            mean_closure_error = (
                complete_mean_delta - expected_complete_mean_delta
            )
            perturbed_mean_values = perturbed_mean_delta.values
            complete_mean_values = complete_mean_delta.values
            expected_complete_mean_values = expected_complete_mean_delta.values
            mean_closure_error_values = mean_closure_error.values
            rows.append({
                "ref_name": cfg["ref_name"],
                "date": cfg["date_tag"],
                "variable": var,
                "mean_member_mean_delta": float(np.mean(member_mean_delta)),
                "mean_member_rms_delta": float(np.mean(member_rms_delta)),
                "min_delta_across_members": float(np.min(member_min_delta)),
                "max_delta_across_members": float(np.max(member_max_delta)),
                "rms_perturbed_mean_delta": float(np.sqrt(np.mean(perturbed_mean_values ** 2))),
                "max_abs_perturbed_mean_delta": float(np.max(np.abs(perturbed_mean_values))),
                "rms_complete_mean_delta": float(np.sqrt(np.mean(complete_mean_values ** 2))),
                "max_abs_complete_mean_delta": float(np.max(np.abs(complete_mean_values))),
                "rms_expected_complete_mean_delta": float(np.sqrt(np.mean(expected_complete_mean_values ** 2))),
                "max_abs_expected_complete_mean_delta": float(np.max(np.abs(expected_complete_mean_values))),
                "rms_mean_closure_error": float(np.sqrt(np.mean(mean_closure_error_values ** 2))),
                "max_abs_mean_closure_error": float(np.max(np.abs(mean_closure_error_values))),
            })
    return pd.DataFrame(rows)

def enforce_validation(validation):
    failures = []
    for row in validation.itertuples(index=False):
        tolerance = ENSEMBLE_MEAN_TOLERANCE[row.variable]
        if row.variable == "Q" and CLIP_Q:
            continue
        if row.max_abs_mean_closure_error > tolerance:
            failures.append(
                f"{row.date} {row.variable}: max_abs_mean_closure_error="
                f"{row.max_abs_mean_closure_error:.6e} > {tolerance:.6e}"
            )
    if failures:
        raise RuntimeError("Written perturbation validation failed:\n" + "\n".join(failures))

def component_files(cfg):
    files = []
    for pattern in COMPONENT_PATTERNS:
        matches = list(cfg["ref_rest_dir"].glob(pattern))
        if not matches:
            raise FileNotFoundError(
                f"No non-atmosphere component files matching {pattern!r} "
                f"in {cfg['ref_rest_dir']}"
            )
        files.extend(matches)
    return sorted(set(files))

def maybe_link_components(cfg):
    if not CREATE_COMPONENT_LINKS:
        return []
    linked_files = []
    for src in component_files(cfg):
        out_file = cfg["output_dir"] / src.name
        if out_file.exists() or out_file.is_symlink():
            if RESUME_EXISTING_OUTPUTS and not OVERWRITE_OUTPUT_FILES:
                if (
                    COMPONENT_LINK_MODE == "symlink"
                    and out_file.is_symlink()
                    and out_file.resolve() == src.resolve()
                ):
                    linked_files.append(out_file)
                    continue
            if not OVERWRITE_OUTPUT_FILES:
                raise FileExistsError(f"Component output already exists: {out_file}")
            out_file.unlink()
        if COMPONENT_LINK_MODE == "symlink":
            out_file.symlink_to(src)
        elif COMPONENT_LINK_MODE == "copy":
            shutil.copy2(src, out_file)
        else:
            raise ValueError("COMPONENT_LINK_MODE must be 'symlink' or 'copy'.")
        linked_files.append(out_file)
    return linked_files

def write_metadata(cfg, written_files, linked_files, control_checksum):
    metadata = {
        "ref_name": cfg["ref_name"],
        "ref_atm_source": cfg["ref_atm_source"],
        "date_tag": cfg["date_tag"],
        "set_name": cfg["set_name"],
        "dart_case_prefix": cfg["dart_case_prefix"],
        "dart_rest_dir": str(cfg["dart_rest_dir"]),
        "dart_diag_dir": str(cfg["dart_diag_dir"]),
        "dart_source_type": cfg["dart_source_type"],
        "dart_mean_file": str(cfg["dart_mean_file"]) if cfg["dart_mean_file"] else None,
        "dart_mean_mode": cfg["dart_mean_mode"],
        "ref_atm_file": str(cfg["ref_atm_file"]),
        "ref_rest_dir": str(cfg["ref_rest_dir"]),
        "method": "selected-subset-centered DART perturbations",
        "selected_dart_member_csv": str(SELECTED_DART_MEMBER_CSV),
        "control_output_member": "EN01",
        "control_sha256": control_checksum,
        "selected_dart_members": selected_dart_members,
        "centering_members": centering_dart_members,
        "perturbation_source_members": perturbation_dart_members,
        "omitted_source_member": omitted_dart_members[0],
        "output_to_dart_member": {
            f"EN{output_member:02d}": f"EN{dart_member:02d}"
            for output_member, dart_member in output_to_dart_member.items()
        },
        "compatible_vars": cfg["compatible_vars"],
        "alpha": ALPHA,
        "clip_q": CLIP_Q,
        "q_min": Q_MIN,
        "output_dir": str(cfg["output_dir"].resolve()),
        "out_case_prefix": cfg["out_case_prefix"],
        "out_atm_stream": cfg["out_atm_stream"],
        "write_output_files": WRITE_OUTPUT_FILES,
        "overwrite_output_files": OVERWRITE_OUTPUT_FILES,
        "resume_existing_outputs": RESUME_EXISTING_OUTPUTS,
        "ensemble_mean_tolerance": ENSEMBLE_MEAN_TOLERANCE,
        "physical_limits": PHYSICAL_LIMITS,
        "warn_q_clip_fraction": WARN_Q_CLIP_FRACTION,
        "enforce_max_q_clip_fraction": ENFORCE_MAX_Q_CLIP_FRACTION,
        "max_q_clip_fraction": MAX_Q_CLIP_FRACTION,
        "created_atmosphere_files": [str(p) for p in written_files],
        "create_component_links": CREATE_COMPONENT_LINKS,
        "component_link_mode": COMPONENT_LINK_MODE,
        "created_component_links": [str(p) for p in linked_files],
        "formula": "A_output = A_REF + alpha*(A_DART_selected - mean(A_DART_selected_20_member_subset))",
        "expected_complete_mean_offset": "-alpha*(A_DART_omitted - mean(A_DART_selected_20_member_subset))/20",
    }
    metadata_file = cfg["output_dir"] / "dart_perturbation_metadata.json"
    with open(metadata_file, "w") as f:
        json.dump(metadata, f, indent=2)
    return metadata_file


## 5. Generate the multi-date HYB perturbation library


In [ ]:
all_diag = []
all_validation = []
all_q_clip = []
metadata_files = []

for cfg in run_configs:
    print("\n" + "=" * 80)
    print(f"Processing {cfg['ref_name']} {cfg['set_name']} {cfg['date_tag']} using {cfg['dart_source_type']}")
    cfg["output_dir"].mkdir(parents=True, exist_ok=True)
    report_variable_storage(cfg["ref_atm_file"], cfg["compatible_vars"])
    complete_member_mapping.to_csv(
        cfg["output_dir"] / "output_to_dart_member_mapping.csv", index=False
    )

    dart_mean = load_dart_mean(cfg, cfg["compatible_vars"])

    diag = perturbation_diagnostics(cfg, dart_mean)
    display(diag)
    diag.to_csv(cfg["output_dir"] / "perturbation_diagnostics.csv", index=False)
    all_diag.append(diag)

    written_files = []
    q_clip_rows = []
    control_checksum = None
    if WRITE_OUTPUT_FILES:
        print(f"Writing {cfg['date_tag']} EN01 exact control")
        control_file, control_checksum = write_control_file(cfg)
        written_files.append(control_file)

        for output_member, dart_member in output_to_dart_member.items():
            print(
                f"Writing {cfg['date_tag']} EN{output_member:02d} "
                f"from DART EN{dart_member:02d}"
            )
            out_file, member_q_clip_rows = write_member_file(
                cfg, output_member, dart_member, dart_mean
            )
            written_files.append(out_file)
            q_clip_rows.extend(member_q_clip_rows)
        if len(written_files) != len(all_output_members):
            raise RuntimeError(
                f"Expected {len(all_output_members)} outputs, wrote {len(written_files)}."
            )
        print(f"Wrote {len(written_files)} atmosphere files to {cfg['output_dir']}")

        q_clip = pd.DataFrame(q_clip_rows)
        display(q_clip)
        q_clip.to_csv(cfg["output_dir"] / "q_clipping_diagnostics.csv", index=False)
        if not q_clip.empty:
            all_q_clip.append(q_clip)

        validation = validate_written_files(cfg, dart_mean)
        validation["ensemble_mean_tolerance"] = validation["variable"].map(
            ENSEMBLE_MEAN_TOLERANCE
        )
        validation["ensemble_mean_tolerance_enforced"] = ~((
            validation["variable"] == "Q"
        ) & CLIP_Q)
        validation["passes_tolerance"] = (
            ~validation["ensemble_mean_tolerance_enforced"]
            | (
                validation["max_abs_mean_closure_error"]
                <= validation["ensemble_mean_tolerance"]
            )
        )
        display(validation)
        validation.to_csv(cfg["output_dir"] / "written_file_validation.csv", index=False)
        enforce_validation(validation)
        all_validation.append(validation)
    else:
        print("WRITE_OUTPUT_FILES=False; no files written.")

    linked_files = maybe_link_components(cfg)
    if linked_files:
        print(f"Created {len(linked_files)} component {COMPONENT_LINK_MODE}s/copies.")
    else:
        print("No component links/copies created.")

    metadata_file = write_metadata(
        cfg, written_files, linked_files, control_checksum
    )
    metadata_files.append(metadata_file)
    print("Metadata written to:", metadata_file)

if all_diag:
    all_diag_df = pd.concat(all_diag, ignore_index=True)
    display(all_diag_df)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_diag = all_diag_df.loc[
            all_diag_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_diag.to_csv(output_root / "all_perturbation_diagnostics.csv", index=False)

if all_validation:
    all_validation_df = pd.concat(all_validation, ignore_index=True)
    display(all_validation_df)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_validation = all_validation_df.loc[
            all_validation_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_validation.to_csv(output_root / "all_written_file_validation.csv", index=False)

if all_q_clip:
    all_q_clip_df = pd.concat(all_q_clip, ignore_index=True)
    for output_root in sorted({cfg["output_root"] for cfg in run_configs}, key=str):
        case_q_clip = all_q_clip_df.loc[
            all_q_clip_df["ref_name"].isin([
                cfg["ref_name"] for cfg in run_configs if cfg["output_root"] == output_root
            ])
        ]
        case_q_clip.to_csv(output_root / "all_q_clipping_diagnostics.csv", index=False)

print("Metadata files:")
for path in metadata_files:
    print(path)


## 6. Experiment usage

This notebook creates a final 20-member atmospheric ensemble for each configured reference case and date. EN01 is an exact unperturbed control. The selected 20-member DART subset defines the mean; EN02-EN20 use anomalies from the first 19 rows of that subset, and metadata records the omitted 20th-row source member. Because one centered anomaly is omitted, the expected complete-ensemble mean offset is \(-\alpha\,\delta A_{\mathrm{omitted}}/20\); validation checks closure against that expected offset.

The output is organized by date, for example:

```text
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_dartpert20_atm_multidate/2011-12-21-00000/
/global/cfs/cdirs/m4849/zhan391/e3sm_dart/free_clim_dartpert20_atm_multidate/2012-05-21-00000/
```

Each date directory contains the case-specific `*.EN01-EN20.<stream>.<date>.nc` files, the output-to-DART mapping, diagnostics, and metadata.
